# TLMAC Integrated Flow

Full pipeline from N2UQ quantised ResNet-18 to FPGA LUT INIT files:

1. Load N2UQ quantised weights from `real_res18.pth.tar`
2. For each 3x3 conv layer: extract unique weight groups
3. Cluster steps that share weight groups (horizontal placement)
4. Simulated annealing for routing reduction (vertical placement)
5. Generate LUT-6 INIT truth tables
6. Export all TLMAC hex files

Reference: Gerlinghoff et al., FPGA 2024 | N2UQ: Liu et al., CVPR 2022

In [ ]:
!apt-get install -y git-lfs > /dev/null 2>&1 && git lfs install
!git clone https://github.com/leozqi/resnet18_n2uq.git || true
%cd resnet18_n2uq
!pip install torch torchvision numpy matplotlib 2>/dev/null

In [ ]:
import math, os, json
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. N2UQ Architecture (inline from resnet.py)

In [ ]:
class LearnableBias(nn.Module):
    def __init__(self, out_chn):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, out_chn, 1, 1))
    def forward(self, x):
        return x + self.bias.expand_as(x)

class LTQ(nn.Module):
    """Learnable Threshold Quantiser from N2UQ."""
    def __init__(self, num_bits):
        super().__init__()
        self.n_val = 2 ** num_bits - 1
        self.interval = 2.0 / self.n_val
        self.start = nn.Parameter(torch.tensor([0.0]))
        self.a = nn.Parameter(torch.tensor([self.interval] * self.n_val))
        self.scale1 = nn.Parameter(torch.tensor([1.0]))
        self.scale2 = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x):
        x = x * self.scale1
        a_pos = torch.where(self.a > 1e-3, self.a, torch.tensor(1e-3))
        step_right = torch.tensor(0.0)
        x_f = x
        x_b = x
        for i in range(self.n_val):
            step_right = step_right + self.interval
            if i == 0:
                th_f = self.start + a_pos[0] / 2
                th_b = self.start
                x_f = torch.where(x > th_f, step_right, torch.tensor(0.0))
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, torch.tensor(0.0))
            else:
                th_f = th_f + a_pos[i - 1] / 2 + a_pos[i] / 2
                th_b = th_b + a_pos[i - 1]
                x_f = torch.where(x > th_f, step_right, x_f)
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, x_b)
        th_b = th_b + a_pos[-1]
        x_b = torch.where(x > th_b, torch.tensor(2.0), x_b)
        return (x_f.detach() + x_b - x_b.detach()) * self.scale2

class HardQuantizeConv(nn.Module):
    """Quantised convolution (HardQuantize) from N2UQ."""
    def __init__(self, in_chn, out_chn, num_bits, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.num_bits = num_bits
        self.kernel_size = kernel_size
        self.clip_val = nn.Parameter(torch.tensor([2.0]))
        self.shape = (out_chn, in_chn, kernel_size, kernel_size)
        self.weight = nn.Parameter((torch.rand(self.shape) - 0.5) * 0.001)

    def forward(self, x):
        real_weights = self.weight
        gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
        sf = gamma * real_weights.abs().mean(dim=[1, 2, 3], keepdim=True)
        sf = sf.detach()
        sw = real_weights / sf
        cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
        n = float(2 ** self.num_bits - 1) / self.clip_val
        qw = sf * (torch.round((cw + self.clip_val / 2) * n) / n - self.clip_val / 2)
        return F.conv2d(x, qw, stride=self.stride, padding=self.padding)

    def get_integer_weights(self):
        """Return quantised integer weights for TLMAC extraction."""
        with torch.no_grad():
            rw = self.weight
            gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
            sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True)
            sw = rw / sf
            cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
            n = float(2 ** self.num_bits - 1) / self.clip_val
            qi = torch.round((cw + self.clip_val / 2) * n)  # 0..2^n-1
        return qi.to(torch.int8)

In [ ]:
N_BIT = 3
QUANTIZE_DOWNSAMPLE = True

class BasicBlockQ(nn.Module):
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bias11 = LearnableBias(inplanes)
        self.prelu1 = nn.PReLU(inplanes)
        self.bias12 = LearnableBias(inplanes)
        self.quan1 = LTQ(N_BIT)
        self.conv1 = HardQuantizeConv(inplanes, planes, N_BIT, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.bias21 = LearnableBias(planes)
        self.prelu2 = nn.PReLU(planes)
        self.bias22 = LearnableBias(planes)
        self.quan2 = LTQ(N_BIT)
        self.conv2 = HardQuantizeConv(planes, planes, N_BIT, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.bias31 = LearnableBias(planes)
        self.prelu3 = nn.PReLU(planes)
        self.bias32 = LearnableBias(planes)

    def forward(self, x):
        identity = x
        out = self.bias12(self.prelu1(self.bias11(x)))
        out = self.bn1(self.conv1(self.quan1(out)))
        out = self.bias22(self.prelu2(self.bias21(out)))
        out = self.bn2(self.conv2(self.quan2(out)))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.bias32(self.prelu3(self.bias31(out)))
        return out

class N2UQ_ResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_block(64, 64, 2)
        self.layer2 = self._make_block(64, 128, 2, stride=2)
        self.layer3 = self._make_block(128, 256, 2, stride=2)
        self.layer4 = self._make_block(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, 1000)

    def _make_block(self, inplanes, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes:
            if QUANTIZE_DOWNSAMPLE:
                downsample = nn.Sequential(
                    LTQ(N_BIT),
                    HardQuantizeConv(inplanes, planes, N_BIT, 1, stride, 0),
                    nn.BatchNorm2d(planes))
            else:
                downsample = nn.Sequential(
                    nn.Conv2d(inplanes, planes, 1, stride, bias=False),
                    nn.BatchNorm2d(planes))
        layers = [BasicBlockQ(inplanes, planes, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(BasicBlockQ(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.fc(torch.flatten(self.avgpool(x), 1))
        return x

model_n2uq = N2UQ_ResNet18().to(device)

CKPT = 'pretrained_models/quantize_downsampling_true/real_res18.pth.tar'
ckpt = torch.load(CKPT, map_location='cpu')
if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']
model_n2uq.load_state_dict(ckpt, strict=False)
model_n2uq.eval()
for p in model_n2uq.parameters(): p.requires_grad_(False)

print(f'N2UQ ResNet-18 loaded: {CKPT}')
print(f'Parameters: {sum(p.numel() for p in model_n2uq.parameters()):,}')

## 2. Identify 3x3 Conv Layers

In [ ]:
conv_layers = []
for name, module in model_n2uq.named_modules():
    if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
        conv_layers.append((name, module))
        print(f'  {name}: {module.shape}  bits={module.num_bits}')
print(f'Total 3x3 conv layers: {len(conv_layers)}')


## 3. TLMAC Parameters & Helpers

G=3 (weight group = one kernel row), BW=3, NCLUS=8 select codes, LUT-6.

In [ ]:
G = 3  # weight group size
BW = N_BIT  # weight bit width
NCLUS = 2 ** (6 - G)  # 8
NLUT_PER_ARR = BW + math.ceil(math.log2(G))  # 5
DK = 3

print(f'TLMAC: G={G}, BW={BW}, NCLUS={NCLUS}, NLUT_PER_ARR={NLUT_PER_ARR}')

def extract_weight_groups(w_q):
    """w_q: [D_o, D_i, 3, 3] int weights.
    Returns wg_assign[D_s, D_p], group_id_map, D_s, D_p."""
    D_o, D_i, _, _ = w_q.shape
    D_s = D_i * DK
    D_p = D_o
    w = w_q.reshape(D_o, D_s, DK).permute(1, 0, 2)  # [D_s, D_p, G]
    gid_map, nxt = {}, 0
    wg = np.full((D_s, D_p), -1, dtype=np.int32)
    for s in range(D_s):
        for p in range(D_p):
            key = tuple(w[s, p].tolist())
            if key not in gid_map:
                gid_map[key] = nxt
                nxt += 1
            wg[s, p] = gid_map[key]
    return wg, gid_map, D_s, D_p

def greedy_cluster(C_mat, nclus):
    """Agglomerative clustering to at most nclus clusters."""
    DS, NG = C_mat.shape
    clusters = [{s} for s in range(DS)]
    while len(clusters) > nclus:
        best, best_sz = (0, 1), float('inf')
        for i in range(len(clusters)):
            for j in range(i + 1, len(clusters)):
                u = set()
                for s in clusters[i] | clusters[j]:
                    for g in range(NG):
                        if C_mat[s, g]: u.add(g)
                if len(u) < best_sz:
                    best_sz = len(u)
                    best = (i, j)
        i, j = best
        merged = clusters[i] | clusters[j]
        clusters = [clusters[k] for k in range(len(clusters)) if k != j]
        clusters[i % len(clusters)] = merged
    cg = []
    for cl in clusters:
        gset = set()
        for s in cl:
            for g in range(NG):
                if C_mat[s, g]: gset.add(g)
        cg.append(gset)
    return clusters, cg

def routing_sa(clusters, wg, group_to_outs, n_arr, dp):
    """SA for routing reduction (Algorithm 1, alpha=1.4)."""
    ITERS = 10000
    ALPHA = 1.4
    cg_outs = []
    for cl in clusters:
        go = defaultdict(set)
        for s in cl:
            for p in range(dp):
                go[wg[s, p]].add(p)
        cg_outs.append(dict(go))
    init_pp = [{gid: np.random.randint(0, n_arr) for gid in go} for go in cg_outs]
    def cnt(pp):
        used = np.zeros((n_arr, dp), dtype=bool)
        for assign in pp:
            for gid, arr in assign.items():
                for p in group_to_outs.get(gid, set()):
                    used[arr, p] = True
        return int(used.sum())
    curr = [dict(a) for a in init_pp]
    curr_r = cnt(curr)
    best = [dict(a) for a in curr]
    best_r = curr_r
    hist = []
    for it in range(1, ITERS + 1):
        T = ITERS / (it + 1) ** ALPHA
        c = np.random.randint(0, len(clusters))
        gids = list(curr[c].keys())
        if len(gids) < 2:
            hist.append((it, curr_r, best_r))
            continue
        i0, i1 = np.random.choice(len(gids), 2, replace=False)
        nn_pp = [dict(a) for a in curr]
        nn_pp[c][gids[i0]], nn_pp[c][gids[i1]] = nn_pp[c][gids[i1]], nn_pp[c][gids[i0]]
        nr = cnt(nn_pp)
        if nr < curr_r or np.random.random() < math.exp(-(nr - curr_r + 1) / max(T, 1e-12)):
            curr, curr_r = nn_pp, nr
            if curr_r < best_r:
                best_r, best = curr_r, [dict(a) for a in curr]
        hist.append((it, curr_r, best_r))
    return best, best_r, hist

## 4. Compile All Layers

In [ ]:
all_results = {}
wg_assign_all = {}  # layer_name -> wg_assign array

for name, module in conv_layers:
    wq = module.get_integer_weights()
    wg, gid_map, D_s, D_p = extract_weight_groups(wq)
    N_UWG = len(gid_map)
    wg_assign_all[name] = wg

    # Build assignment matrix and cluster
    C = np.zeros((D_s, N_UWG), dtype=np.int8)
    for s in range(D_s):
        for p in range(D_p):
            C[s, wg[s, p]] = 1
    clusters, cg = greedy_cluster(C, NCLUS)
    n_arr = max(len(g) for g in cg)

    # Group-to-outputs mapping
    g2o = defaultdict(set)
    for s in range(D_s):
        for p in range(D_p):
            g2o[wg[s, p]].add(p)
    g2o = dict(g2o)

    # Routing SA
    best_pp, best_r, hist = routing_sa(clusters, wg, g2o, n_arr, D_p)

    # Step-to-cluster mapping
    stc = {}
    for ci, cl in enumerate(clusters):
        for s in cl:
            stc[s] = ci

    # LUT INIT generation
    id2g = {v: k for k, v in gid_map.items()}
    all_init = {}
    for ci in range(len(clusters)):
        for ai in range(n_arr):
            sg = {}
            for gid, arr in best_pp[ci].items():
                if arr == ai:
                    sg[len(sg)] = gid
            zgid = gid_map.get((0,) * G, 0)
            while len(sg) < NCLUS:
                sg[len(sg)] = zgid
            iv = [0] * NLUT_PER_ARR
            for addr in range(64):
                sel = (addr >> G) & (NCLUS - 1)
                abits = addr & ((1 << G) - 1)
                wg_t = id2g.get(sg.get(sel, zgid), (0,) * G)
                mac = sum(((abits >> gi) & 1) * wg_t[gi] for gi in range(G))
                if mac < 0: mac += (1 << NLUT_PER_ARR)
                for k in range(NLUT_PER_ARR):
                    iv[k] |= ((mac >> k) & 1) << addr
            all_init[(ci, ai)] = iv

    all_results[name] = dict(
        D_s=D_s, D_p=D_p, N_UWG=N_UWG,
        n_clusters=len(clusters), n_arr=n_arr,
        best_routes=best_r, possible=n_arr * D_p,
        all_init=all_init, step_to_cluster=stc,
        best_placement=best_pp, history=hist,
        id_to_group=id2g, gid_map=gid_map,
    )
    route_pct = 1.0 - best_r / (n_arr * D_p) if n_arr * D_p > 0 else 0.0
    short = name.replace('.conv', '.c')
    print(f'{short}: {N_UWG} uwg, {len(clusters)} cl, {n_arr} arr, '
          f'routes {best_r}/{n_arr*D_p} ({route_pct:.0%} red)')

print(f'Compiled {len(all_results)} layers')

## 5. Routing SA Convergence

In [ ]:
n_layers = len(all_results)
if n_layers == 0:
    print('No layers compiled — check previous cell for errors.')
else:
    ncols = min(5, n_layers)
    nrows = (n_layers + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    if n_layers == 1:
        axes = np.array([axes])
    axes_flat = np.array(axes).flatten() if hasattr(axes, 'flatten') else [axes]
    for idx, (name, res) in enumerate(all_results.items()):
        ax = axes_flat[idx]
        iters = [h[0] for h in res['history']]
        curr  = [h[1] for h in res['history']]
        best  = [h[2] for h in res['history']]
        ax.plot(iters, curr, alpha=0.3, linewidth=0.5)
        ax.plot(iters, best, linewidth=1.5)
        short = name.replace('.conv', '.c')
        ax.set_title(f"{short}\n{res['N_UWG']} grps, {res['n_arr']} arr", fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n_layers, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
total_lut6 = 0
print(f"{'Layer':<30s} {'UWG':>5s} {'Cl':>4s} {'N_arr':>5s} {'LUT-6':>7s} {'Routes':>10s}")
print('-' * 65)
for name, res in all_results.items():
    lut6 = res['n_arr'] * NLUT_PER_ARR * res['n_clusters']
    total_lut6 += lut6
    short = name.replace('.conv', '.c')
    print(f"{short:<30s} {res['N_UWG']:>5d} {res['n_clusters']:>4d} {res['n_arr']:>5d} {lut6:>7d} {res['best_routes']:>10d}")
print('-' * 65)
print(f'Total LUT-6: {total_lut6:,}')

## 6. Export TLMAC Hex Files

Writes to `tlmac_output/weights/{layer}/`.
Files match the `INIT_FILE` parameter of `lut_array.sv` and `cluster_map_rom.sv`.

In [ ]:
OUT_DIR = 'tlmac_output'
os.makedirs(os.path.join(OUT_DIR, 'weights'), exist_ok=True)
metadata = {}

for name, res in all_results.items():
    safe = name.replace('.', '_')
    ldir = os.path.join(OUT_DIR, 'weights', safe)
    os.makedirs(ldir, exist_ok=True)
    D_s = res['D_s']
    n_arr = res['n_arr']

    # cluster_map.hex: one line per step
    with open(os.path.join(ldir, 'cluster_map.hex'), 'w') as f:
        for s in range(D_s):
            f.write(f"{res['step_to_cluster'].get(s, 0):02X}\n")

    # lut_arrN.hex: NLUT_PER_ARR lines per cluster, one file per array
    for ai in range(n_arr):
        with open(os.path.join(ldir, f'lut_arr{ai}.hex'), 'w') as f:
            for ci in range(res['n_clusters']):
                for val in res['all_init'][(ci, ai)]:
                    f.write(f"{val:016X}\n")

    # routing_bitmap.hex
    wg = wg_assign_all[name]
    g2o = defaultdict(set)
    for s in range(D_s):
        for p in range(res['D_p']):
            g2o[wg[s, p]].add(p)
    R = np.zeros((n_arr, res['D_p']), dtype=bool)
    for ci, assign in enumerate(res['best_placement']):
        for gid, ai in assign.items():
            for p in g2o.get(gid, set()):
                R[ai, p] = True
    with open(os.path.join(ldir, 'routing_bitmap.hex'), 'w') as f:
        for e in range(n_arr):
            bitmap = 0
            for p in range(res['D_p']):
                if R[e, p]: bitmap |= (1 << p)
            nw = (res['D_p'] + 63) // 64
            for wi in range(nw):
                f.write(f"{(bitmap >> (wi * 64)) & 0xFFFFFFFFFFFFFFFF:016X}\n")

    metadata[safe] = {
        'layer': name, 'D_s': D_s, 'D_p': res['D_p'],
        'N_UWG': res['N_UWG'], 'clusters': res['n_clusters'],
        'n_arr': n_arr, 'G': G, 'BW': BW, 'NLUT_PER_ARR': NLUT_PER_ARR, 'NCLUS': NCLUS,
        'best_routes': res['best_routes'], 'possible': res['possible'],
    }

with open(os.path.join(OUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Exported {len(metadata)} layers to {OUT_DIR}/')
for k in metadata:
    v = metadata[k]
    print(f"  {k}: {v['n_arr']} arr, {v['clusters']} cl, {v['best_routes']} routes")

# List output files
for root, dirs, files in os.walk(os.path.join(OUT_DIR, 'weights')):
    for fname in sorted(files):
        sz = os.path.getsize(os.path.join(root, fname))
        rel = os.path.relpath(os.path.join(root, fname), OUT_DIR)
        print(f'  {rel}  ({sz:,} bytes)')

In [ ]:
print('\n=== TLMAC Integrated Compilation Summary ===')
print(f'Model: N2UQ ResNet-18 ({N_BIT}-bit, quantize_downsample={QUANTIZE_DOWNSAMPLE})')
print(f'Layers compiled:       {len(all_results)}')
total_groups = sum(r['N_UWG'] for r in all_results.values())
print(f'Total unique groups:   {total_groups:,}')
total_arr = sum(r['n_arr'] for r in all_results.values())
print(f'Total LUT arrays:      {total_arr}')
print(f'Total LUT-6:           {total_lut6:,}')
print(f'Output:                {OUT_DIR}/')